In [1]:
!apt update -q && apt-get install -qy openjdk-21-jdk-headless > /dev/null 2>&1

import subprocess
from IPython.core.magic import register_cell_magic

class PersistentJShell:
    def __init__(self):
        self.proc = subprocess.Popen(
            ["jshell", "-s"],
            stdin=subprocess.PIPE,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1
        )

    def execute(self, code):
        if self.proc.poll() is not None:
            raise RuntimeError("JShell process has terminated.")

        marker = "___JSHELL_CELL_END___"
        payload = code.strip() + f'\nSystem.out.println("{marker}");\n'
        self.proc.stdin.write(payload)
        self.proc.stdin.flush()

        lines = []
        while True:
            line = self.proc.stdout.readline()
            if not line:
                break
            if marker in line:
                break
            lines.append(line)

        return "".join(lines)

_jshell_instance = PersistentJShell()

@register_cell_magic
def jshell(line, cell):
    out = _jshell_instance.execute(cell)
    if out:
        print(out, end="")


Get:1 https://cloud.r-project.org/bin/linux/ubuntu noble-cran40/ InRelease [3,631 B]
Get:2 https://cli.github.com/packages stable InRelease [3,917 B]
Hit:3 http://archive.ubuntu.com/ubuntu noble InRelease
Get:4 http://archive.ubuntu.com/ubuntu noble-updates InRelease [126 kB]
Get:5 http://security.ubuntu.com/ubuntu noble-security InRelease [126 kB]
Get:6 https://r2u.stat.illinois.edu/ubuntu noble InRelease [9,161 B]
Get:7 https://cloud.r-project.org/bin/linux/ubuntu noble-cran40/ Packages [73.2 kB]
Get:8 http://archive.ubuntu.com/ubuntu noble-backports InRelease [126 kB]
Get:9 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu noble InRelease [17.8 kB]
Get:10 https://r2u.stat.illinois.edu/ubuntu noble/main all Packages [10.2 MB]
Get:11 http://archive.ubuntu.com/ubuntu noble-updates/restricted amd64 Packages [2,048 kB]
Get:12 http://security.ubuntu.com/ubuntu noble-security/main amd64 Packages [1,300 kB]
Get:13 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu noble/main amd64 

In [2]:
%%jshell

void runPy(String code) throws Exception {
    String marker = "___END_OF_PYTHON_CELL___";

    // Base64 encode code payload to bypass all quote & newline syntax errors
    String b64 = java.util.Base64.getEncoder().encodeToString(code.getBytes(java.nio.charset.StandardCharsets.UTF_8));

    // Guarantee marker printing using try...finally and display trailing expressions using ast
    String payload = String.format(
        "import base64, ast, traceback\n" +
        "try:\n" +
        "    code_str = base64.b64decode('%s').decode('utf-8')\n" +
        "    tree = ast.parse(code_str)\n" +
        "    if tree.body and isinstance(tree.body[-1], ast.Expr):\n" +
        "        exec(compile(ast.Module(body=tree.body[:-1], type_ignores=[]), '<string>', 'exec'), globals())\n" +
        "        res = eval(compile(ast.Expression(body=tree.body[-1].value), '<string>', 'eval'), globals())\n" +
        "        if res is not None:\n" +
        "            print(res)\n" +
        "    else:\n" +
        "        exec(compile(tree, '<string>', 'exec'), globals())\n" +
        "except Exception:\n" +
        "    traceback.print_exc()\n" +
        "finally:\n" +
        "    print('%s', flush=True)\n",
        b64, marker
    );

    pyIn.println(payload);

    String line;
    while ((line = pyOut.readLine()) != null) {
        if (line.contains(marker)) break;
        System.out.println(line);
    }
}

import java.io.BufferedReader;
import java.io.InputStreamReader;
import java.io.PrintStream;
import java.io.IOException;

Process pythonProcess;
PrintStream pyIn;
BufferedReader pyOut;

try {
    ProcessBuilder pb = new ProcessBuilder("python3", "-i", "-u");
    // Redirect stderr to stdout to prevent pipe deadlock and capture Python errors
    pb.redirectErrorStream(true);
    pythonProcess = pb.start();

    pyIn = new PrintStream(pythonProcess.getOutputStream(), true);
    pyOut = new BufferedReader(new InputStreamReader(pythonProcess.getInputStream()));

    System.out.println("Python interpreter initialized safely.");
} catch (IOException e) {
    e.printStackTrace();
}

[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
Sep 28, 2026 11:05:15 AM java.util.prefs.FileSystemPreferences$1 run
INFO: Created user preferences directory.
->  void runPy(String code) throws Exception {
>>          String marker = "___END_OF_PYTHON_CELL___";
>>     
>>         // Base64 encode code payload to bypass all quote & newline syntax errors
>>          String b64 = java.util.Base64.getEncoder().encodeToString(code.getBytes(java.nio.charset.StandardCharsets.UTF_8));
>>     
>>          // Guarantee marker printing using try...finally and display trailing expressions using ast
>>         String payload = String.format(
>>              "import base64, ast, traceback\n" +
>>             "try:\n" +
>>  

In [3]:
%%jshell

runPy("""
# Google Colab setup: fetch this repository and use this notebook's directory.
from pathlib import Path ; from IPython.display import display, HTML
import os

REPO_ROOT = Path('/content/BITS_programming')
if not REPO_ROOT.exists():os.system('git clone https://github.com/aqwertyuiop48/BITS_programming.git /content/BITS_programming');

NOTEBOOK_DIR = REPO_ROOT / 'assignments/assignment_5'
os.chdir(NOTEBOOK_DIR)
print(f'Working directory: {NOTEBOOK_DIR}')
""");


->  Python 3.13.15 (main, Aug  6 2026, 11:06:22) [GCC 13.3.0] on linux
Type "help", "copyright", "credits" or "license" for more information.
>>> >>> ... ... ... ... ... ... ... ... ... ... ... ... ... ... Cloning into '/content/BITS_programming'...
Updating files:   4% (70/1417)
Updating files:   5% (71/1417)
Updating files:   6% (86/1417)
Updating files:   7% (100/1417)
Updating files:   8% (114/1417)
Updating files:   9% (128/1417)
Updating files:  10% (142/1417)
Updating files:  11% (156/1417)
Updating files:  11% (164/1417)
Updating files:  12% (171/1417)
Updating files:  13% (185/1417)
Updating files:  14% (199/1417)
Updating files:  15% (213/1417)
Updating files:  16% (227/1417)
Updating files:  17% (241/1417)
Updating files:  18% (256/1417)
Updating files:  19% (270/1417)
Updating files:  20% (284/1417)
Updating files:  21% (298/1417)
Updating files:  22% (312/1417)
Updating files:  23% (326/1417)
Updating files:  24% (341/1417)
Updating files:  25% (355/1417)
Updating files:  

In [4]:
%%jshell

runPy("""
# Data Aggregation Techniques
#
# In this lab we'll learn about various data aggregation techniques through
# different scenarios to understand the practical applications of these methods
""");


In [5]:
%%jshell

runPy("""
# Download the dataset file to local storage to current directory (from publicly
# shared drive file)
#
# Original source:
# https://www.kaggle.com/competitions/house-prices-advanced-regression-techniques
os.system('gdown 1Yhw8tBCmjYrDZ5Rm1xPrP2y5bmBmxoEP');
""");


->  >>> >>> ... ... ... ... ... ... ... ... ... ... ... ... ... ... Downloading...
From: https://drive.google.com/uc?id=1Yhw8tBCmjYrDZ5Rm1xPrP2y5bmBmxoEP
To: /content/BITS_programming/assignments/assignment_5/housing.csv

  0%|                                               | 0.00/461k [00:00<?, ?B/s]
100%|███████████████████████████████████████| 461k/461k [00:00<00:00, 7.47MB/s]
0


In [6]:
%%jshell

runPy("""
# Import required libraries
import pandas as pd

from sklearn.preprocessing import MinMaxScaler
from sklearn.preprocessing import StandardScaler
import math # use
""");


In [7]:
%%jshell

runPy("""
# Load the dataset file into memory
DATASET_PATH = "./housing.csv"
df = pd.read_csv(DATASET_PATH)
df
""");


->  >>> >>> ... ... ... ... ... ... ... ... ... ... ... ... ... ...         Id  MSSubClass MSZoning  ...  SaleType  SaleCondition SalePrice
0        1          60       RL  ...        WD         Normal    208500
1        2          20       RL  ...        WD         Normal    181500
2        3          60       RL  ...        WD         Normal    223500
3        4          70       RL  ...        WD        Abnorml    140000
4        5          60       RL  ...        WD         Normal    250000
...    ...         ...      ...  ...       ...            ...       ...
1455  1456          60       RL  ...        WD         Normal    175000
1456  1457          20       RL  ...        WD         Normal    210000
1457  1458          70       RL  ...        WD         Normal    266500
1458  1459          20       RL  ...        WD         Normal    142125
1459  1460          20       RL  ...        WD         Normal    147500

[1460 rows x 81 columns]


In [8]:
%%jshell

runPy("""
# We'll be performing aggregation functions on both grouped and un-grouped data
#
# For grouped data, we'll be considering the neighborhood for our aggregation
# based analysis ahead
#
grouped_df = df.groupby('Neighborhood')
grouped_df.describe()
""");


->  >>> >>> ... ... ... ... ... ... ... ... ... ... ... ... ... ...                  Id                          ... SalePrice                     
              count        mean         std  ...       50%        75%       max
Neighborhood                                 ...                               
Blmngtn        17.0  773.941176  370.871688  ...  191000.0  213490.00  264561.0
Blueste         2.0  778.500000  252.437121  ...  137500.0  144250.00  151000.0
BrDale         16.0  719.187500  443.481412  ...  106000.0  118000.00  125000.0
BrkSide        58.0  734.724138  435.850884  ...  124300.0  141175.00  223500.0
ClearCr        28.0  604.928571  382.491339  ...  200250.0  242225.00  328000.0
CollgCr       150.0  726.840000  445.513942  ...  197200.0  225725.00  424870.0
Crawfor        51.0  801.078431  432.064247  ...  200624.0  239000.00  392500.0
Edwards       100.0  762.930000  413.290620  ...  121750.0  145225.00  320000.0
Gilbert        79.0  749.151899  426.017488  ...  18

In [9]:
%%jshell

runPy("""
# Total home valuation of all houses in dataset
#
# Scenario: A real estate developer wants to know the total value of sales in
# general to gauge market activity and investment potential
total_valuation = df['SalePrice'].sum()
print("Total Valuation of houses: USD", total_valuation)
print()

# Total home valuation of each neighborhood
#
# Scenario: A real estate developer wants to know the total value of sales in
# each neighborhood to gauge market activity and investment potential
neighborhood_valuation = grouped_df['SalePrice'].sum()
neighborhood_valuation.name = "HomeValuation"
neighborhood_valuation
""");


->  >>> >>> ... ... ... ... ... ... ... ... ... ... ... ... ... ... Total Valuation of houses: USD 264144946

Neighborhood
Blmngtn     3312805
Blueste      275000
BrDale      1671900
BrkSide     7240375
ClearCr     5951832
CollgCr    29694866
Crawfor    10741861
Edwards    12821970
Gilbert    15235506
IDOTRR      3704580
MeadowV     1675800
Mitchel     7657236
NAmes      32815593
NPkVill     1284250
NWAmes     13800655
NoRidge    13747108
NridgHt    24352838
OldTown    14489459
SWISU       3564784
Sawyer     10122692
SawyerW    11006792
Somerst    19382666
StoneBr     7762475
Timber      9205403
Veenker     2626500
Name: HomeValuation, dtype: int64


In [10]:
%%jshell

runPy("""
# Average home price in the dataset
#
# Scenario: As a buyer I might want to know the average home prices to get a
# rough idea on the kind of budget I should have while going through the prices
average_price = df['SalePrice'].mean()
print("Average Price of houses: USD", average_price)

# Average home price in each neighborhood
#
# Scenario: As a buyer I might want to see the average home price in a given
# neighborhood to prioritize on those neighborhoods which are likely to be
# within my budget
average_price_neighborhood = grouped_df['SalePrice'].mean()
average_price_neighborhood.name = "AveragePrice"
average_price_neighborhood
""");


->  >>> >>> ... ... ... ... ... ... ... ... ... ... ... ... ... ... Average Price of houses: USD 180921.19589041095
Neighborhood
Blmngtn    194870.882353
Blueste    137500.000000
BrDale     104493.750000
BrkSide    124834.051724
ClearCr    212565.428571
CollgCr    197965.773333
Crawfor    210624.725490
Edwards    128219.700000
Gilbert    192854.506329
IDOTRR     100123.783784
MeadowV     98576.470588
Mitchel    156270.122449
NAmes      145847.080000
NPkVill    142694.444444
NWAmes     189050.068493
NoRidge    335295.317073
NridgHt    316270.623377
OldTown    128225.300885
SWISU      142591.360000
Sawyer     136793.135135
SawyerW    186555.796610
Somerst    225379.837209
StoneBr    310499.000000
Timber     242247.447368
Veenker    238772.727273
Name: AveragePrice, dtype: float64


In [11]:
%%jshell

runPy("""
# Median home price in dataset
#
# Scenario: As a buyer I might want to know the average home prices to get a
# rough idea on the kind of budget I should have while going through the prices.
# (Similar to mean but using median here as an average instead)
median_price = df['SalePrice'].median()
print("Median Price of homes: USD", median_price)

# Median home price in each neighborhood
#
# Scenario: As a buyer I might want to know the average home prices to get a
# rough idea on the kind of budget I should have while going through the prices.
# (Similar to mean but using median here as an average instead)
median_price_neighborhood = grouped_df['SalePrice'].median()
median_price_neighborhood.name = "MedianPrice"
median_price_neighborhood
""");


->  >>> >>> ... ... ... ... ... ... ... ... ... ... ... ... ... ... Median Price of homes: USD 163000.0
Neighborhood
Blmngtn    191000.0
Blueste    137500.0
BrDale     106000.0
BrkSide    124300.0
ClearCr    200250.0
CollgCr    197200.0
Crawfor    200624.0
Edwards    121750.0
Gilbert    181000.0
IDOTRR     103000.0
MeadowV     88000.0
Mitchel    153500.0
NAmes      140000.0
NPkVill    146000.0
NWAmes     182900.0
NoRidge    301500.0
NridgHt    315000.0
OldTown    119000.0
SWISU      139500.0
Sawyer     135000.0
SawyerW    179900.0
Somerst    225500.0
StoneBr    278000.0
Timber     228475.0
Veenker    218000.0
Name: MedianPrice, dtype: float64


In [12]:
%%jshell

runPy("""
# Minimum sale price in dataset
#
# To know the cheapest home in the dataset
min_price = df['SalePrice'].min()
print("Minimum Price of homes: USD", min_price)

# Minimum sale price in each neighborhood
#
# To know the cheapest home in each neighborhood
min_price_neighborhood = grouped_df['SalePrice'].min()
min_price_neighborhood.name = "MinPrice"
min_price_neighborhood
""");


->  >>> >>> ... ... ... ... ... ... ... ... ... ... ... ... ... ... Minimum Price of homes: USD 34900
Neighborhood
Blmngtn    159895
Blueste    124000
BrDale      83000
BrkSide     39300
ClearCr    130000
CollgCr    110000
Crawfor     90350
Edwards     58500
Gilbert    141000
IDOTRR      34900
MeadowV     75000
Mitchel     84500
NAmes       87500
NPkVill    127500
NWAmes      82500
NoRidge    190000
NridgHt    154000
OldTown     37900
SWISU       60000
Sawyer      62383
SawyerW     76000
Somerst    144152
StoneBr    170000
Timber     137500
Veenker    162500
Name: MinPrice, dtype: int64


In [13]:
%%jshell

runPy("""
# Maximum sale price in dataset
#
# Scenario: A financially well off person wanting to know the most expensive
# home in the dataset to get an idea of the facilities they can get there
max_price = df['SalePrice'].max()
print("Maximum Price of homes: USD", max_price)

# Maximum sale price in each neighborhood
#
# Scenario: A person trying to know the most expensive house in each
# neighborhood
max_price_neighborhood = grouped_df['SalePrice'].max()
max_price_neighborhood.name = "Max Price"
max_price_neighborhood
""");


->  >>> >>> ... ... ... ... ... ... ... ... ... ... ... ... ... ... Maximum Price of homes: USD 755000
Neighborhood
Blmngtn    264561
Blueste    151000
BrDale     125000
BrkSide    223500
ClearCr    328000
CollgCr    424870
Crawfor    392500
Edwards    320000
Gilbert    377500
IDOTRR     169500
MeadowV    151400
Mitchel    271000
NAmes      345000
NPkVill    155000
NWAmes     299800
NoRidge    755000
NridgHt    611657
OldTown    475000
SWISU      200000
Sawyer     190000
SawyerW    320000
Somerst    423000
StoneBr    556581
Timber     378500
Veenker    385000
Name: Max Price, dtype: int64


In [14]:
%%jshell

runPy("""
# Scenario: To know the number of homes in the dataset
num_homes = df['SalePrice'].count()
print("Number of homes sold: ", num_homes)

# Number of homes present in each neighborhood
num_homes_neighborhood = grouped_df['SalePrice'].count()
num_homes_neighborhood.name = "No. of Homes"
num_homes_neighborhood
""");


->  >>> >>> ... ... ... ... ... ... ... ... ... ... ... ... ... ... Number of homes sold:  1460
Neighborhood
Blmngtn     17
Blueste      2
BrDale      16
BrkSide     58
ClearCr     28
CollgCr    150
Crawfor     51
Edwards    100
Gilbert     79
IDOTRR      37
MeadowV     17
Mitchel     49
NAmes      225
NPkVill      9
NWAmes      73
NoRidge     41
NridgHt     77
OldTown    113
SWISU       25
Sawyer      74
SawyerW     59
Somerst     86
StoneBr     25
Timber      38
Veenker     11
Name: No. of Homes, dtype: int64


In [15]:
%%jshell

runPy("""
# Standard deviation of all prices in the dataset
#
# Scenario: A person viewing the dataset might want to get an idea of how
# much do the prices in the dataset vary in general
std_price = df['SalePrice'].std()
print("Standard Deviation of prices: USD", std_price)

# Standard deviation of prices in each neighborhood
#
# Scenario: A financial analyst might want to know the volatility of prices
# in a neighborhood to know how similar the house are there
std_price_neighborhood = grouped_df['SalePrice'].std()
std_price_neighborhood.name = "SD"
std_price_neighborhood
""");


->  >>> >>> ... ... ... ... ... ... ... ... ... ... ... ... ... ... Standard Deviation of prices: USD 79442.50288288663
Neighborhood
Blmngtn     30393.229219
Blueste     19091.883092
BrDale      14330.176493
BrkSide     40348.689270
ClearCr     50231.538993
CollgCr     51403.666438
Crawfor     68866.395472
Edwards     43208.616459
Gilbert     35986.779085
IDOTRR      33376.710117
MeadowV     23491.049610
Mitchel     36486.625334
NAmes       33075.345450
NPkVill      9377.314529
NWAmes      37172.218106
NoRidge    121412.658640
NridgHt     96392.544954
OldTown     52650.583185
SWISU       32622.917679
Sawyer      22345.129157
SawyerW     55651.997820
Somerst     56177.555888
StoneBr    112969.676640
Timber      64845.651549
Veenker     72369.317959
Name: SD, dtype: float64


In [16]:
%%jshell

runPy("""
# Variance of prices in dataset
#
# Similar to standard deviation, but variance is more often used for its math
# property of additivity of variance. Standard deviation is more interpretable
# in general otherwise.
var_price = df['SalePrice'].var()
print("Variance of prices: USD", var_price)

# Variance of prices in each neighborhood
#
# Similar to standard deviation, but variance is more often used for its math
# property of additivity of variance. Standard deviation is more interpretable
# in general otherwise.
var_price_neighborhood = grouped_df['SalePrice'].var()
var_price_neighborhood.name = "Variance"
var_price_neighborhood
""");


->  >>> >>> ... ... ... ... ... ... ... ... ... ... ... ... ... ... Variance of prices: USD 6311111264.297451
Neighborhood
Blmngtn    9.237484e+08
Blueste    3.645000e+08
BrDale     2.053540e+08
BrkSide    1.628017e+09
ClearCr    2.523208e+09
CollgCr    2.642337e+09
Crawfor    4.742580e+09
Edwards    1.866985e+09
Gilbert    1.295048e+09
IDOTRR     1.114005e+09
MeadowV    5.518294e+08
Mitchel    1.331274e+09
NAmes      1.093978e+09
NPkVill    8.793403e+07
NWAmes     1.381774e+09
NoRidge    1.474103e+10
NridgHt    9.291523e+09
OldTown    2.772084e+09
SWISU      1.064255e+09
Sawyer     4.993048e+08
SawyerW    3.097145e+09
Somerst    3.155918e+09
StoneBr    1.276215e+10
Timber     4.204959e+09
Veenker    5.237318e+09
Name: Variance, dtype: float64


In [24]:
%%jshell

runPy("""
# Get the number of neighborhoods in the data
print("Number of neighborhoods in the dataset:", df["Neighborhood"].nunique())

# Checking the possibility of an sell out event in a neighborhood for further
# investigation
print("\\nNumber of houses sold in different years in neighborhood:")
grouped_df["YrSold"].nunique()
""");


->  >>> >>> ... ... ... ... ... ... ... ... ... ... ... ... ... ... Number of neighborhoods in the dataset: 25

Number of houses sold in different years in neighborhood:
Neighborhood
Blmngtn    5
Blueste    2
BrDale     5
BrkSide    5
ClearCr    5
CollgCr    5
Crawfor    5
Edwards    5
Gilbert    5
IDOTRR     5
MeadowV    5
Mitchel    5
NAmes      5
NPkVill    4
NWAmes     5
NoRidge    5
NridgHt    5
OldTown    5
SWISU      5
Sawyer     5
SawyerW    5
Somerst    5
StoneBr    5
Timber     5
Veenker    3
Name: YrSold, dtype: int64


In [25]:
%%jshell

runPy("""
# Get a rough idea of how old streets (can be used in conjuction with
# filtering when years are above/below threshold of interest)
first_sale_event = grouped_df["YearBuilt"].min()
first_sale_event.name = "FirstSaleEventYear"
display(first_sale_event)

# Similarly, one would want to investigate about most recent events so they
# could
last_sale_event = grouped_df["YearBuilt"].max()
last_sale_event.name = "LatestSaleEventYear"
display(last_sale_event)
""");


->  >>> >>> ... ... ... ... ... ... ... ... ... ... ... ... ... ... Neighborhood
Blmngtn    2003
Blueste    1980
BrDale     1970
BrkSide    1908
ClearCr    1908
CollgCr    1969
Crawfor    1910
Edwards    1900
Gilbert    1948
IDOTRR     1900
MeadowV    1970
Mitchel    1940
NAmes      1918
NPkVill    1974
NWAmes     1963
NoRidge    1990
NridgHt    2003
OldTown    1872
SWISU      1911
Sawyer     1875
SawyerW    1928
Somerst    1997
StoneBr    1984
Timber     1948
Veenker    1974
Name: FirstSaleEventYear, dtype: int64
Neighborhood
Blmngtn    2007
Blueste    1980
BrDale     1973
BrkSide    1970
ClearCr    1997
CollgCr    2009
Crawfor    2008
Edwards    2009
Gilbert    2007
IDOTRR     1963
MeadowV    1977
Mitchel    2007
NAmes      1998
NPkVill    1978
NWAmes     2001
NoRidge    2001
NridgHt    2009
OldTown    2003
SWISU      1940
Sawyer     1982
SawyerW    2005
Somerst    2009
StoneBr    2010
Timber     2009
Veenker    1996
Name: LatestSaleEventYear, dtype: int64


In [26]:
%%jshell

runPy("""
# Calculating
# Scenario: A loan officer wants to understand the central tendency of home prices in a way that is robust to outliers, helping them assess risk.
quantile_result = grouped_df['SalePrice'].quantile(0.5)
display(quantile_result)
""");


->  >>> >>> ... ... ... ... ... ... ... ... ... ... ... ... ... ... Neighborhood
Blmngtn    191000.0
Blueste    137500.0
BrDale     106000.0
BrkSide    124300.0
ClearCr    200250.0
CollgCr    197200.0
Crawfor    200624.0
Edwards    121750.0
Gilbert    181000.0
IDOTRR     103000.0
MeadowV     88000.0
Mitchel    153500.0
NAmes      140000.0
NPkVill    146000.0
NWAmes     182900.0
NoRidge    301500.0
NridgHt    315000.0
OldTown    119000.0
SWISU      139500.0
Sawyer     135000.0
SawyerW    179900.0
Somerst    225500.0
StoneBr    278000.0
Timber     228475.0
Veenker    218000.0
Name: SalePrice, dtype: float64


In [27]:
%%jshell

runPy("""
# There are different ways one could aggregate multiple aggregation functions
""");


In [28]:
%%jshell

runPy("""
# Applying Multiple Aggregations to a Single Column
#
# Scenario: A real estate analyst needs a single summary table to see the total,
# average, and maximum sale price for each neighborhood for a quarterly market report.
multiple_agg_result = grouped_df['SalePrice'].agg(['sum', 'mean', 'max'])
multiple_agg_result
""");


->  >>> >>> ... ... ... ... ... ... ... ... ... ... ... ... ... ...                    sum           mean     max
Neighborhood                                 
Blmngtn        3312805  194870.882353  264561
Blueste         275000  137500.000000  151000
BrDale         1671900  104493.750000  125000
BrkSide        7240375  124834.051724  223500
ClearCr        5951832  212565.428571  328000
CollgCr       29694866  197965.773333  424870
Crawfor       10741861  210624.725490  392500
Edwards       12821970  128219.700000  320000
Gilbert       15235506  192854.506329  377500
IDOTRR         3704580  100123.783784  169500
MeadowV        1675800   98576.470588  151400
Mitchel        7657236  156270.122449  271000
NAmes         32815593  145847.080000  345000
NPkVill        1284250  142694.444444  155000
NWAmes        13800655  189050.068493  299800
NoRidge       13747108  335295.317073  755000
NridgHt       24352838  316270.623377  611657
OldTown       14489459  128225.300885  475000
SWISU       

In [29]:
%%jshell

runPy("""
# Applying Different Aggregations to Different Columns
#
# Scenario: A performance dashboard needs to display the total and average home
# price for each neighborhood while also showing the minimum and maximum number
# of bedrooms sold in a single, comprehensive table.
different_agg_result = grouped_df.agg({
    'SalePrice': ['sum', 'mean'], # Total, mean
    'YrSold': ['min', 'max'], # Sales activity duration
})
different_agg_result
""");


->  >>> >>> ... ... ... ... ... ... ... ... ... ... ... ... ... ...              SalePrice                YrSold      
                   sum           mean    min   max
Neighborhood                                      
Blmngtn        3312805  194870.882353   2006  2010
Blueste         275000  137500.000000   2008  2009
BrDale         1671900  104493.750000   2006  2010
BrkSide        7240375  124834.051724   2006  2010
ClearCr        5951832  212565.428571   2006  2010
CollgCr       29694866  197965.773333   2006  2010
Crawfor       10741861  210624.725490   2006  2010
Edwards       12821970  128219.700000   2006  2010
Gilbert       15235506  192854.506329   2006  2010
IDOTRR         3704580  100123.783784   2006  2010
MeadowV        1675800   98576.470588   2006  2010
Mitchel        7657236  156270.122449   2006  2010
NAmes         32815593  145847.080000   2006  2010
NPkVill        1284250  142694.444444   2007  2010
NWAmes        13800655  189050.068493   2006  2010
NoRidge       

In [30]:
%%jshell

runPy("""
import datetime, pytz;
print("Current Time in IST:", datetime.datetime.now(pytz.utc).astimezone(pytz.timezone('Asia/Kolkata')).strftime('%Y-%m-%d %H:%M:%S'))
""");


->  >>> >>> ... ... ... ... ... ... ... ... ... ... ... ... ... ... Current Time in IST: 2026-09-28 16:38:27
